# 19 — Binary reduction with three-class evaluation

This notebook presents the full counterfactual requested: remove every `functional needs repair` case from each training partition, rebuild a broad binary functional/non-functional model screen, reconstruct bounded ensembles, and assess the selected binary recipe against unchanged three-class validation and local-test targets.

**Result:** the selected equal Random-Forest/LightGBM vote improves conditional functional/non-functional accuracy, but reaches only **80.086%** development accuracy and **79.992%** local-test accuracy. Repair recall is necessarily zero, so the accepted three-class workflow remains selected.

## Course-aligned lifecycle

| Step | Application |
| --- | --- |
| 1. Define the goal and scope | Test whether ignoring repair cases improves accuracy on the original task. |
| 2. Gather the data | Reuse the validated labelled modelling data; do not use competition rows. |
| 3. Explore the data | Quantify class prevalence, the binary model's hard accuracy ceiling and class-level errors. |
| 4. Clean and preprocess the data | Reuse established fold-fitted preprocessing. |
| 5. Select and engineer features | Hold the accepted feature policy fixed. |
| 6. Define the machine-learning task | Fit two-class models, then score their two labels against all three target classes. |
| 7. Partition the data | Keep five frozen development folds; remove repair rows only from each training fold, never validation or test. |
| 8. Select and train candidate methods | Compare eleven standalone families and nine fixed soft votes. |
| 9. Evaluate and interpret the results | Prioritise full three-class accuracy; diagnose conditional accuracy, recall, probability quality and confusion. |
| 10. Deploy and iterate | Do not deploy or submit because binary reduction trails the accepted three-class model. |

In [ ]:
from pathlib import Path
import sys
import joblib
import pandas as pd

STAGE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SRC_DIR = STAGE_DIR / 'src'
PROJECT_DIR = STAGE_DIR.parent
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

result = joblib.load(
    PROJECT_DIR / '.runtime' / 'binary-reduction-screen'
    / 'binary-reduction-screen.joblib'
)
result['standalone_summary'].loc[:, [
    'model_name',
    'mean_three_class_accuracy',
    'mean_conditional_binary_accuracy',
    'functional_recall',
    'non_functional_recall',
    'conditional_log_loss',
]]

## Contract and hard ceiling

The repair rows are excluded only from the current training partition. Every validation row remains present. Each candidate emits functional and non-functional probabilities; the aligned repair probability is exactly zero. This avoids the misleading shortcut of deleting repair rows before validation.

Development contains 3,454 repair rows out of 47,520, so any binary-only classifier has a three-class accuracy ceiling of **92.7315%**. All reported three-class scores include those guaranteed errors.

In [ ]:
selected = result['ensembles'][result['selected_key']]
assert result['selected_key'] == 'rf_lgb_equal'
assert selected.out_of_fold_probabilities['functional needs repair'].eq(0).all()
assert result['local_test'].metrics['repair_recall'] == 0
assert result['local_test'].probabilities['functional needs repair'].eq(0).all()

pd.Series({
    'development rows': len(selected.out_of_fold_probabilities),
    'development repair rows': 3454,
    'three-class accuracy ceiling': 1 - 3454 / len(selected.out_of_fold_probabilities),
    'selected binary recipe': result['selected_key'],
})

## Ensemble reconstruction

XGBoost leads the standalone screen at 79.718% three-class accuracy. The equal Random-Forest/LightGBM vote improves it by 0.368 points, wins all five folds and has no negative fold. This passes the established internal gate and freezes the recipe for the local-test assessment.

This is selection *within* the binary experiment. It does not imply that the binary winner beats the accepted three-class workflow.

In [ ]:
result['combined_summary'].loc[result['ensembles'].keys(), [
    'model_name',
    'mean_three_class_accuracy',
    'mean_conditional_binary_accuracy',
    'accuracy_change',
    'fold_wins',
    'worst_fold_change',
    'passes_gate',
]].sort_values('mean_three_class_accuracy', ascending=False)

## Comparison with the accepted three-class vote

The binary vote is better at the remaining two-class boundary: conditional accuracy rises from 85.290% to 86.364%, producing 473 additional correct functional/non-functional decisions. It abandons 1,204 repair decisions that the accepted model gets right. The net loss is 731 rows, reducing full development accuracy from **81.625% to 80.086%**.

In [ ]:
accepted = joblib.load(
    PROJECT_DIR / '.runtime' / 'geography-screen' / 'frozen-baseline.joblib'
).blend
accepted_confusion = accepted.confusion_counts
accepted_nonrepair_correct = (
    accepted_confusion.loc['functional', 'functional']
    + accepted_confusion.loc['non functional', 'non functional']
)
binary_confusion = selected.confusion_counts
binary_nonrepair_correct = (
    binary_confusion.loc['functional', 'functional']
    + binary_confusion.loc['non functional', 'non functional']
)
nonrepair_rows = (
    accepted_confusion.loc['functional'].sum()
    + accepted_confusion.loc['non functional'].sum()
)

pd.DataFrame({
    'accepted three-class': [
        accepted.metric_summary.loc['accuracy', 'mean'],
        accepted_nonrepair_correct / nonrepair_rows,
        accepted_nonrepair_correct,
        accepted_confusion.loc['functional needs repair', 'functional needs repair'],
    ],
    'selected binary': [
        result['combined_summary'].loc[result['selected_key'], 'mean_three_class_accuracy'],
        result['combined_summary'].loc[result['selected_key'], 'mean_conditional_binary_accuracy'],
        binary_nonrepair_correct,
        0,
    ],
}, index=[
    'full three-class accuracy',
    'conditional binary accuracy',
    'correct non-repair rows',
    'correct repair rows',
])

## Local-test assessment

After selection, the equal vote was refitted on all 44,066 non-repair development rows and scored against all 11,880 local-test labels. It reaches **79.992%** full accuracy and **86.258%** conditional binary accuracy, with 91.414% functional recall, 78.970% non-functional recall and zero repair recall.

The earlier three-class Random-Forest/histogram workflow scored 80.825% on this partition. Binary reduction makes 178 more correct non-repair decisions, then loses the 277 repair decisions the earlier workflow got right: a net **0.833-point loss**. This local test has been opened by earlier work, so the result is a consistency check rather than fresh independent validation.

In [ ]:
display(result['local_test'].metrics.to_frame('value'))
display(result['local_test'].confusion_counts)

## Decision

Retain three-class training and the accepted 55% XGBoost / 45% Random Forest vote. The repair class is difficult but still contributes materially more correct decisions than binary reduction recovers elsewhere. Stop this competition-accuracy loop without producing a competition prediction.

A binary model could still suit a different operational workflow in which repairable cases have already been triaged. That would require a newly defined target and success measure; it is not evidence for changing this three-class competition model.

The full candidate table, confusion decomposition and reproduction command are in [`../reports/binary-reduction-screen.md`](../reports/binary-reduction-screen.md).